In [7]:
# Projeto Crypto Analytics

In [1]:
import datetime
import requests
import pandas as pd
import sqlite3
import os

In [2]:
from dotenv import load_dotenv

load_dotenv()

coingecko_apikey = os.getenv("coingecko_apikey")

requisicao = requests.get(f"https://api.coingecko.com/api/v3/simple/price?ids=bitcoin,ethereum,solana,ripples,cardano,binancecoin&vs_currencies=usd,eur,brl&include_market_cap=true&include_24hr_change=true/x-cg-demo-api-key:{coingecko_apikey}")

df_requisicao = pd.DataFrame(requisicao.json())

display(df_requisicao)

,bitcoin,ethereum,solana,ripples,cardano,binancecoin
usd,8.405600e+04,2.673030e+03,1.216800e+02,0.001972,2.548940e-01,7.776100e+02
usd_market_cap,1.688689e+12,3.263364e+11,7.148547e+10,167627.656675,9.561205e+09,1.035570e+11
eur,7.382900e+04,2.347810e+03,1.068800e+02,0.001732,2.238820e-01,6.830000e+02
eur_market_cap,1.483231e+12,2.866320e+11,6.278804e+10,147232.902570,8.397922e+09,9.095750e+10
brl,4.362610e+05,1.387331e+04,6.315600e+02,0.010235,1.320000e+00,4.035880e+03
brl_market_cap,8.764462e+12,1.693718e+12,3.710167e+11,870004.300909,4.962361e+10,5.374710e+11


In [3]:
tickers = {
    "Bitcoin":"BTC",
    "Ethereum":"ETH",
    "Solana":"SOL",
    "Ripples":"XRP",
    "Cardano":"ADA",
    "Binancecoin":"BNB"
}

df_ajustado = df_requisicao.stack(0).unstack(0)
df_ajustado["data_coleta"] = datetime.datetime.now()
df_ajustado = df_ajustado.reset_index(drop=False)
df_ajustado.rename(columns={"index":"coin"}, inplace=True)
df_ajustado["coin"] = df_ajustado["coin"].str.title()
df_ajustado["ticker"] = df_ajustado["coin"].map(tickers)

display(df_ajustado)
df_ajustado.dtypes

,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta,ticker
0,Bitcoin,84056.000000,1.688689e+12,73829.000000,1.483231e+12,436261.000000,8.764462e+12,2026-09-27 22:05:56.999619,BTC
1,Ethereum,2673.030000,3.263364e+11,2347.810000,2.866320e+11,13873.310000,1.693718e+12,2026-09-27 22:05:56.999619,ETH
2,Solana,121.680000,7.148547e+10,106.880000,6.278804e+10,631.560000,3.710167e+11,2026-09-27 22:05:56.999619,SOL
3,Ripples,0.001972,1.676277e+05,0.001732,1.472329e+05,0.010235,8.700043e+05,2026-09-27 22:05:56.999619,XRP
4,Cardano,0.254894,9.561205e+09,0.223882,8.397922e+09,1.320000,4.962361e+10,2026-09-27 22:05:56.999619,ADA
5,Binancecoin,777.610000,1.035570e+11,683.000000,9.095750e+10,4035.880000,5.374710e+11,2026-09-27 22:05:56.999619,BNB


coin                         str
usd                      float64
usd_market_cap           float64
eur                      float64
eur_market_cap           float64
brl                      float64
brl_market_cap           float64
data_coleta       datetime64[us]
ticker                       str
dtype: object

In [4]:
conexao = sqlite3.connect("crypto_analytics.db")

df_ajustado.to_sql(name="tb_crypto_base_api", con=conexao, if_exists="append", index=False)

conexao.close()